# DistilBERT fine-tune for News Bias Classifier (Colab)

Run this notebook on Colab with **Runtime > Change runtime type > T4 GPU**.

**Inputs you upload from your local machine:**
- `train.parquet`
- `val.parquet`
- `test.parquet`

These are produced locally by `python -m src.data.preprocess`.

**Output:** `distilbert.zip` (~260 MB) is downloaded automatically. Unzip into your local `models/` so you have `models/distilbert/`.

Total runtime on a free T4: ~20-30 min.

## 1. Confirm GPU is attached

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU and re-run.'

## 2. Install pinned deps
Colab usually has these, but pin versions to avoid surprise breakage.

In [ ]:
!pip install -q 'transformers>=4.40' 'datasets>=2.18' 'accelerate>=0.29' 'scikit-learn>=1.4' pyarrow

## 3. Upload your parquet files
When the file picker pops up, select `train.parquet`, `val.parquet`, and `test.parquet` from your local `data/processed/` folder. Multi-select is fine.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print(name, len(uploaded[name]) // 1024, 'KB')
for required in ('train.parquet', 'val.parquet', 'test.parquet'):
    assert required in uploaded, f'Missing {required} — please re-run this cell and select all three files.'

## 4. Load data

In [ ]:
import pandas as pd
from datasets import Dataset

LABELS = ['left', 'center', 'right']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def load(path):
    df = pd.read_parquet(path)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df

train_df = load('train.parquet')
val_df = load('val.parquet')
test_df = load('test.parquet')
print(f'train={len(train_df)} val={len(val_df)} test={len(test_df)}')
print('train label balance:', train_df['label'].value_counts().to_dict())

def to_hf(df):
    return Dataset.from_pandas(
        df[['text', 'label_id']].rename(columns={'label_id': 'labels'}),
        preserve_index=False,
    )

train_ds = to_hf(train_df)
val_ds = to_hf(val_df)
test_ds = to_hf(test_df)

## 5. Tokenize

In [ ]:
import re
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 512

# --- Inline cleaner: must match src/data/clean.py exactly so train/inference agree.
OUTLET_NAMES = {
    'cnn','msnbc','fox','foxnews','fox news','npr','monitor','csmonitor',
    'reuters','ap','associated press','bloomberg','bbc','afp',
    'nyt','new york times','wsj','wall street journal',
    'washingtonpost','washington post','wapo',
    'politico','axios','the hill','atlantic','slate','salon',
    'huffpost','huffington post','vox','intercept','buzzfeed','vice',
    'breitbart','newsmax','dailywire','daily wire','daily caller',
    'the federalist','national review','thehill','guardian','telegraph',
    'usa today','usatoday',
}
_OUTLET = re.compile(r'\b(?:' + '|'.join(sorted(OUTLET_NAMES, key=len, reverse=True)) + r')\b', re.IGNORECASE)
_BOILERPLATE = re.compile('|'.join([
    r'story highlights?', r'replay\s+more\s+videos?\s+must\s+watch',
    r'more\s+videos?\s+must\s+watch', r'more\s+videos?', r'must\s+watch',
    r'just\s+watched', r'\breplay\b',
    r'delivered\s+to\s+(?:your\s+)?inbox',
    r'sign\s+up\s+for\s+(?:our\s+)?newsletter',
    r'subscribe\s+to\s+(?:our\s+)?newsletter',
    r'privacy\s+policy', r'by\s+signing\s+up', r'you\s+agree\s+to\s+our',
    r'©\s*\d{4}.*?reserved', r'all\s+rights\s+reserved',
    r'click\s+here\s+to', r'follow\s+us\s+on\s+(?:twitter|facebook)',
    r'read\s+more\s*:', r'^\s*\(reuters\)\s*[-—]\s*',
]), re.IGNORECASE)
_WS = re.compile(r'\s+')

def clean_for_modeling(text: str) -> str:
    if not text:
        return ''
    s = _BOILERPLATE.sub(' ', text)
    s = _OUTLET.sub(' ', s)
    return _WS.sub(' ', s).strip().lower()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    cleaned = [clean_for_modeling(t) for t in batch['text']]
    return tokenizer(cleaned, truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=['text'])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=['text'])

## 6. Train

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

OUTPUT_DIR = 'models/distilbert'
EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='_runs',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    fp16=True,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_out = trainer.predict(test_ds)
preds = np.argmax(preds_out.predictions, axis=-1)
print(classification_report(preds_out.label_ids, preds, target_names=LABELS, digits=4, zero_division=0))
print('\nConfusion matrix (rows=true, cols=pred):')
print(pd.DataFrame(
    confusion_matrix(preds_out.label_ids, preds, labels=list(range(len(LABELS)))),
    index=LABELS, columns=LABELS,
))

## 8. Save and zip the trained model

In [ ]:
import os, shutil
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.make_archive('distilbert', 'zip', root_dir='.', base_dir=OUTPUT_DIR)
print('Created distilbert.zip')
!ls -lh distilbert.zip

## 9. Download the zip
Your browser will save `distilbert.zip` (~260 MB). Then locally:

```bash
cd "c:/Users/ruben/Desktop/25-26 Sem 2/Advanced AI/bias classifier"
unzip distilbert.zip   # produces models/distilbert/
```

(On Windows without `unzip`, right-click the zip > Extract All > choose the project folder.)

In [ ]:
from google.colab import files
files.download('distilbert.zip')